# AstroCLIMB — Qwen3-VL-8B relation-focused auxiliary QLoRA

This validation experiment targets the two weak classes, `same_paper` and
`related_papers`. It constructs 20,000 auxiliary examples from the public
`adsabs/AstroCLIMB` metadata, then continues training on the permanent
9,200/800 Kaggle split.

The model never receives DOI, title, author, UUID, reference, or citing fields.
Those fields are used only to generate labels and choose difficult examples.

| Auxiliary class | Label | Rows |
|---|---:|---:|
| `same_figure` | 0 | 2,000 |
| `same_paper` | 1 | 6,000 |
| `related_papers` | 2 | 8,000 |
| `unrelated_papers` | 3 | 4,000 |

Select **GPU T4 x2**, enable Internet for the Hugging Face metadata API, and
attach the AstroCLIMB Kaggle competition dataset. Test inference is deliberately
absent: this notebook selects a configuration on validation only.


In [ ]:
# Preserve Kaggle's torch stack while installing the tested training libraries.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes==0.47.0"


In [ ]:
import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import subprocess
import sys
import time
import unicodedata
import urllib.parse
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)
print('torch:', torch.__version__)
print('CUDA has not been initialized:', not torch.cuda.is_initialized())


In [ ]:
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
DIGIT_TO_LABEL = dict(enumerate(TARGET_COLUMNS))

VAL_PER_CLASS = 200
EXPECTED_TRAIN_ROWS = 9200
EXPECTED_VALIDATION_ROWS = 800
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
GRADIENT_ACCUMULATION = 8

AUXILIARY_COUNTS = {0: 2000, 1: 6000, 2: 8000, 3: 4000}
EXPECTED_AUXILIARY_ROWS = sum(AUXILIARY_COUNTS.values())
HF_DATASET = 'adsabs/AstroCLIMB'
HF_PAGE_SIZE = 100
HF_METADATA_ROWS = 25000  # Sufficient for the 20K pilot; avoids 943 fragile viewer requests.

REBUILD_KAGGLE_CACHE = False
REFETCH_HF_METADATA = False
REBUILD_AUXILIARY = False
RUN_AUXILIARY_STAGE = True
RUN_COMPETITION_STAGE = True

WORK_ROOT = Path('/kaggle/working/astroclimb_relation_auxiliary') if Path('/kaggle/working').exists() else Path('./astroclimb_relation_auxiliary')
IMAGE_ROOT = WORK_ROOT / 'images'
TRAIN_MANIFEST = WORK_ROOT / 'train_9200.jsonl'
VALIDATION_MANIFEST = WORK_ROOT / 'validation_800.jsonl'
HF_METADATA_PATH = WORK_ROOT / 'astroclimb_hf_metadata.jsonl'
AUXILIARY_MANIFEST = WORK_ROOT / 'auxiliary_20000.jsonl'
AUXILIARY_ADAPTER = WORK_ROOT / 'auxiliary_adapter'
FINAL_ADAPTER = WORK_ROOT / 'best_relation_adapter'
AUXILIARY_WORK = WORK_ROOT / 'auxiliary_stage'
FINAL_WORK = WORK_ROOT / 'competition_stage'
for path in [WORK_ROOT, IMAGE_ROOT, AUXILIARY_WORK, FINAL_WORK]:
    path.mkdir(parents=True, exist_ok=True)


def locate_csv(filename):
    for candidate in [
        Path('/kaggle/input/competitions/astroclimb') / filename,
        Path('/kaggle/input/astroclimb') / filename,
    ]:
        if candidate.exists():
            return candidate
    roots = [Path('/kaggle/input'), Path('data')]
    candidates = [p for root in roots if root.exists() for p in root.rglob(filename)]
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'{filename} not found. Attach the AstroCLIMB competition data.')
    return candidates[0]


def locate_model():
    if Path('/kaggle/input').exists():
        configs = list(Path('/kaggle/input').rglob('config.json'))
        candidates = [p.parent for p in configs if 'qwen3' in str(p).lower() and 'vl' in str(p).lower() and '8b' in str(p).lower()]
        if candidates:
            return str(sorted(candidates, key=lambda p: len(str(p)))[0])
    return MODEL_ID


TRAIN_CSV = locate_csv('train.csv')
TEST_CSV = locate_csv('test.csv')
MODEL_PATH = locate_model()
print('Train:', TRAIN_CSV)
print('Test:', TEST_CSV)
print('Model:', MODEL_PATH)
print('Working directory:', WORK_ROOT)


## Permanent balanced validation split

This reproduces the seed-42 split used by the existing 8B restricted-loss
baseline: 200 examples per class are held out, leaving class counts
`[800, 2800, 2800, 2800]` for competition adaptation.


In [ ]:
def get_label(row):
    values = [int(float(row[column])) for column in TARGET_COLUMNS]
    if sum(values) != 1:
        raise ValueError(f'Invalid one-hot label for id={row.get("id")}: {values}')
    return values.index(1)


def select_validation_ids(path, per_class=200, seed=42):
    rng = random.Random(seed)
    reservoirs = {label: [] for label in range(4)}
    seen = {label: 0 for label in range(4)}
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row_number, row in enumerate(reader, start=1):
            label = get_label(row)
            seen[label] += 1
            bucket = reservoirs[label]
            if len(bucket) < per_class:
                bucket.append(row['id'])
            else:
                position = rng.randrange(seen[label])
                if position < per_class:
                    bucket[position] = row['id']
            if row_number % 1000 == 0:
                print('Split scan:', row_number)
    selected = {identifier for ids in reservoirs.values() for identifier in ids}
    print('Rows seen:', {DIGIT_TO_LABEL[k]: v for k, v in seen.items()})
    print('Validation IDs:', len(selected))
    assert len(selected) == EXPECTED_VALIDATION_ROWS
    return selected


validation_ids = select_validation_ids(TRAIN_CSV, VAL_PER_CLASS, SEED)


## Cache the Kaggle training and validation objects

The same-figure replay examples come only from the 9,200-row training split.
No validation object is copied into the auxiliary manifest.


In [ ]:
def looks_like_image(value):
    return isinstance(value, str) and value.lstrip().startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image'))


def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    image = Image.open(io.BytesIO(base64.b64decode(value, validate=False)))
    image.load()
    return image.convert('RGB')


def resize_to_area(image, max_pixels=MAX_PIXELS):
    width, height = image.size
    if width * height <= max_pixels:
        return image
    scale = math.sqrt(max_pixels / (width * height))
    return image.resize((max(1, round(width * scale)), max(1, round(height * scale))), Image.Resampling.LANCZOS)


def cache_object(value):
    if not looks_like_image(value):
        return {'kind': 'caption', 'value': value}
    digest = hashlib.sha256(value.encode('utf-8')).hexdigest()
    path = IMAGE_ROOT / f'{digest}.png'
    if not path.exists():
        resize_to_area(decode_image(value)).save(path, format='PNG', compress_level=3)
    return {'kind': 'image', 'value': str(path)}


def count_lines(path):
    if not path.exists():
        return -1
    with path.open('r', encoding='utf-8') as handle:
        return sum(1 for _ in handle)


def kaggle_manifests_complete():
    return count_lines(TRAIN_MANIFEST) == EXPECTED_TRAIN_ROWS and count_lines(VALIDATION_MANIFEST) == EXPECTED_VALIDATION_ROWS


def build_kaggle_manifests():
    counts = {'train': 0, 'validation': 0}
    class_counts = {'train': [0] * 4, 'validation': [0] * 4}
    modality_counts = {'train': Counter(), 'validation': Counter()}
    started = time.perf_counter()
    with (
        TRAIN_CSV.open('r', encoding='utf-8', newline='') as source,
        TRAIN_MANIFEST.open('w', encoding='utf-8') as train_output,
        VALIDATION_MANIFEST.open('w', encoding='utf-8') as validation_output,
    ):
        for index, row in enumerate(csv.DictReader(source), start=1):
            label = get_label(row)
            obj_1, obj_2 = cache_object(row['obj_1']), cache_object(row['obj_2'])
            modality = obj_1['kind'][0].upper() + obj_2['kind'][0].upper()
            split = 'validation' if row['id'] in validation_ids else 'train'
            record = {'id': row['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'label': label, 'modality': modality}
            (validation_output if split == 'validation' else train_output).write(json.dumps(record, ensure_ascii=False) + '\n')
            counts[split] += 1
            class_counts[split][label] += 1
            modality_counts[split][modality] += 1
            if index % 250 == 0:
                print(f'Kaggle preprocessing {index}/10000 | {(time.perf_counter()-started)/60:.2f} min')
    assert counts == {'train': EXPECTED_TRAIN_ROWS, 'validation': EXPECTED_VALIDATION_ROWS}
    assert class_counts['validation'] == [VAL_PER_CLASS] * 4
    print('Rows:', counts)
    print('Class counts:', class_counts)
    print('Modality counts:', {k: dict(v) for k, v in modality_counts.items()})


if REBUILD_KAGGLE_CACHE or not kaggle_manifests_complete():
    build_kaggle_manifests()
else:
    print('Reusing cached Kaggle manifests.')
print('Manifest rows:', count_lines(TRAIN_MANIFEST), count_lines(VALIDATION_MANIFEST))
print('Cached PNGs:', len(list(IMAGE_ROOT.glob('*.png'))))
gc.collect()


## Fetch lightweight Hugging Face metadata

The rows API returns captions and paper metadata without downloading the full
72 GB image corpus. The pilot reads 25,000 metadata rows in resumable batches of
100 rather than scanning all 94,233 rows through the fragile viewer service. It uses
the public metadata only to construct labels; auxiliary class 0 is replayed from
the Kaggle training split.


In [ ]:
def normalize_doi(value):
    value = unicodedata.normalize('NFKC', str(value or '')).strip().lower()
    for prefix in ('https://doi.org/', 'http://doi.org/', 'doi:'):
        if value.startswith(prefix):
            value = value[len(prefix):]
    return value.strip().rstrip('.,;')


def normalize_caption(value):
    value = unicodedata.normalize('NFKC', str(value or '')).lower()
    return re.sub(r'\s+', ' ', value).strip()


def fetch_json(url, retries=30):
    for attempt in range(retries):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'AstroCLIMB-relation-pilot/1.0'})
            with urllib.request.urlopen(request, timeout=90) as response:
                return json.load(response)
        except Exception as error:
            if attempt + 1 == retries:
                raise
            delay = min(60, 2 ** min(attempt, 6)) + random.random()
            print(f'API retry {attempt + 1}/{retries} after {error!r}; sleeping {delay}s')
            time.sleep(delay)


def metadata_api_url(offset, length):
    query = urllib.parse.urlencode({
        'dataset': HF_DATASET,
        'config': 'default',
        'split': 'train',
        'offset': offset,
        'length': length,
    })
    return 'https://datasets-server.huggingface.co/rows?' + query


def fetch_hf_metadata():
    if REFETCH_HF_METADATA and HF_METADATA_PATH.exists():
        HF_METADATA_PATH.unlink()
    existing = max(0, count_lines(HF_METADATA_PATH))
    if existing >= HF_METADATA_ROWS:
        print('Reusing sufficient cached HF metadata:', existing)
        return existing
    offset = existing
    mode = 'a' if offset else 'w'
    with HF_METADATA_PATH.open(mode, encoding='utf-8') as output:
        while offset < HF_METADATA_ROWS:
            request_length = min(HF_PAGE_SIZE, HF_METADATA_ROWS - offset)
            payload = fetch_json(metadata_api_url(offset, request_length))
            available = int(payload['num_rows_total'])
            if HF_METADATA_ROWS > available:
                raise RuntimeError(f'Requested {HF_METADATA_ROWS} rows but the dataset has {available}.')
            page = payload['rows']
            if not page:
                raise RuntimeError(f'Empty metadata page at offset {offset}')
            for item in page:
                row = item['row']
                compact = {
                    'row_idx': int(item['row_idx']),
                    'uuid': str(row.get('UUID') or ''),
                    'doi': normalize_doi(row.get('Paper DOI')),
                    'title': str(row.get('Paper Title') or ''),
                    'caption': str(row.get('Image Caption') or ''),
                    'authors': str(row.get('Image Authors') or ''),
                    'references': [normalize_doi(x) for x in row.get('References DOIs') or [] if normalize_doi(x)],
                    'citing': [normalize_doi(x) for x in row.get('Citing DOIs') or [] if normalize_doi(x)],
                }
                output.write(json.dumps(compact, ensure_ascii=False) + '\n')
            offset += len(page)
            output.flush()  # Make every successful page resumable after a later HTTP failure.
            if offset % 1000 == 0 or offset == HF_METADATA_ROWS:
                os.fsync(output.fileno())
                print(f'HF metadata: {offset}/{HF_METADATA_ROWS}')
            time.sleep(0.10)  # Avoid hammering the public rows service.
    downloaded = count_lines(HF_METADATA_PATH)
    assert downloaded >= HF_METADATA_ROWS
    return downloaded


total_hf_rows = fetch_hf_metadata()
assert total_hf_rows > EXPECTED_AUXILIARY_ROWS


## Exclude known validation and test papers

Exact normalized caption matches recover paper DOIs for caption-containing
Kaggle rows. Every recovered validation or test DOI is excluded from auxiliary
training. Image-only Kaggle rows cannot be mapped without downloading the full
public image corpus; the notebook reports this limitation explicitly.


In [ ]:
caption_to_dois = defaultdict(set)
hf_rows = []
with HF_METADATA_PATH.open('r', encoding='utf-8') as handle:
    for metadata_index, line in enumerate(handle):
        if metadata_index >= HF_METADATA_ROWS:
            break
        row = json.loads(line)
        if row['doi'] and row['caption']:
            hf_rows.append(row)
            caption_to_dois[normalize_caption(row['caption'])].add(row['doi'])
print('Usable HF caption rows:', len(hf_rows))


def captions_from_manifest(path):
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            for key in ('obj_1', 'obj_2'):
                if row[key]['kind'] == 'caption':
                    yield row[key]['value']


excluded_dois = set()
matched_validation_captions = 0
for caption in captions_from_manifest(VALIDATION_MANIFEST):
    matches = caption_to_dois.get(normalize_caption(caption), set())
    matched_validation_captions += int(bool(matches))
    excluded_dois.update(matches)

matched_test_captions = 0
test_caption_count = 0
with TEST_CSV.open('r', encoding='utf-8', newline='') as handle:
    for row in csv.DictReader(handle):
        for key in ('obj_1', 'obj_2'):
            if not looks_like_image(row[key]):
                test_caption_count += 1
                matches = caption_to_dois.get(normalize_caption(row[key]), set())
                matched_test_captions += int(bool(matches))
                excluded_dois.update(matches)

print('Matched validation captions:', matched_validation_captions)
print('Matched test captions:', matched_test_captions, '/', test_caption_count)
print('Excluded validation/test DOIs:', len(excluded_dois))
print('Limitation: image-only validation/test papers are not recoverable from caption matching.')


## Construct the 20K auxiliary manifest

Classes 1–3 are caption-caption examples so the pilot emphasizes scientific
relationships and remains computationally feasible. Class 0 replays genuine
same-figure examples from the Kaggle training split.


In [ ]:
STOPWORDS = {
    'a', 'an', 'and', 'as', 'at', 'by', 'for', 'from', 'in', 'into', 'of', 'on',
    'or', 'the', 'to', 'using', 'with', 'study', 'new', 'observations', 'analysis',
    'results', 'properties', 'toward', 'towards', 'through', 'our', 'we', 'i', 'ii',
}


def title_tokens(text):
    return {
        token for token in re.findall(r'[a-z0-9]+', normalize_caption(text))
        if len(token) >= 4 and token not in STOPWORDS
    }


def caption_object(text):
    return {'kind': 'caption', 'value': text}


def relation_record(identifier, row_a, row_b, label):
    return {
        'id': identifier,
        'obj_1': caption_object(row_a['caption']),
        'obj_2': caption_object(row_b['caption']),
        'label': int(label),
        'modality': 'CC',
    }


def write_auxiliary_manifest():
    rng = random.Random(SEED)
    eligible = [row for row in hf_rows if row['doi'] not in excluded_dois]
    papers = defaultdict(list)
    paper_meta = {}
    for row in eligible:
        papers[row['doi']].append(row)
        meta = paper_meta.setdefault(row['doi'], {'title': row['title'], 'neighbors': set()})
        meta['neighbors'].update(row['references'])
        meta['neighbors'].update(row['citing'])

    examples = []

    # Label 0: replay only training examples, sampling with replacement when needed.
    same_figure_rows = []
    with TRAIN_MANIFEST.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            if int(row['label']) == 0:
                same_figure_rows.append(row)
    assert same_figure_rows
    for index in range(AUXILIARY_COUNTS[0]):
        source = dict(rng.choice(same_figure_rows))
        source['id'] = f'aux-same-figure-{index}'
        examples.append(source)

    # Label 1: distinct figures/captions from the same DOI.
    multi_figure_dois = [doi for doi, rows in papers.items() if len(rows) >= 2]
    seen = set()
    while len(seen) < AUXILIARY_COUNTS[1]:
        doi = rng.choice(multi_figure_dois)
        row_a, row_b = rng.sample(papers[doi], 2)
        key = tuple(sorted((row_a['uuid'], row_b['uuid'])))
        if key in seen:
            continue
        seen.add(key)
        examples.append(relation_record(f'aux-same-paper-{len(seen)-1}', row_a, row_b, 1))

    # Label 2: direct reference/citing edges, treated symmetrically.
    paper_dois = set(papers)
    edges = set()
    for doi, meta in paper_meta.items():
        for neighbor in meta['neighbors']:
            if neighbor in paper_dois and neighbor != doi:
                edges.add(tuple(sorted((doi, neighbor))))
    edges = sorted(edges)
    if not edges:
        raise RuntimeError('No citation edges were recovered from the eligible HF papers.')
    seen = set()
    attempts = 0
    while len(seen) < AUXILIARY_COUNTS[2]:
        attempts += 1
        doi_a, doi_b = rng.choice(edges)
        row_a, row_b = rng.choice(papers[doi_a]), rng.choice(papers[doi_b])
        key = (row_a['uuid'], row_b['uuid'])
        if key in seen:
            if attempts > AUXILIARY_COUNTS[2] * 100:
                raise RuntimeError('Not enough unique citation-related caption pairs.')
            continue
        seen.add(key)
        examples.append(relation_record(f'aux-related-{len(seen)-1}', row_a, row_b, 2))

    # Label 3: hard negatives share a meaningful title token but no direct edge.
    tokens_by_doi = {doi: title_tokens(meta['title']) for doi, meta in paper_meta.items()}
    inverted = defaultdict(list)
    for doi, tokens in tokens_by_doi.items():
        for token in tokens:
            inverted[token].append(doi)
    useful_tokens = [token for token, dois in inverted.items() if 2 <= len(dois) <= 500]
    seen = set()
    attempts = 0
    while len(seen) < AUXILIARY_COUNTS[3]:
        attempts += 1
        token = rng.choice(useful_tokens)
        doi_a, doi_b = rng.sample(inverted[token], 2)
        if doi_b in paper_meta[doi_a]['neighbors'] or doi_a in paper_meta[doi_b]['neighbors']:
            continue
        row_a, row_b = rng.choice(papers[doi_a]), rng.choice(papers[doi_b])
        key = tuple(sorted((row_a['uuid'], row_b['uuid'])))
        if key in seen:
            if attempts > AUXILIARY_COUNTS[3] * 200:
                raise RuntimeError('Not enough unique hard-unrelated caption pairs.')
            continue
        seen.add(key)
        examples.append(relation_record(f'aux-unrelated-{len(seen)-1}', row_a, row_b, 3))

    rng.shuffle(examples)
    counts = Counter(int(row['label']) for row in examples)
    assert counts == Counter(AUXILIARY_COUNTS)
    assert len(examples) == EXPECTED_AUXILIARY_ROWS
    with AUXILIARY_MANIFEST.open('w', encoding='utf-8') as output:
        for row in examples:
            output.write(json.dumps(row, ensure_ascii=False) + '\n')
    print('Auxiliary rows:', len(examples))
    print('Auxiliary class counts:', dict(sorted(counts.items())))
    print('Eligible papers:', len(papers), '| citation edges:', len(edges))


if REBUILD_AUXILIARY or count_lines(AUXILIARY_MANIFEST) != EXPECTED_AUXILIARY_ROWS:
    write_auxiliary_manifest()
else:
    print('Reusing complete auxiliary manifest.')

with AUXILIARY_MANIFEST.open('r', encoding='utf-8') as handle:
    auxiliary_counts = Counter(json.loads(line)['label'] for line in handle)
assert auxiliary_counts == Counter(AUXILIARY_COUNTS)
print('Verified auxiliary counts:', dict(sorted(auxiliary_counts.items())))


## Two-stage restricted-loss worker

Stage 1 trains one epoch on the 20K auxiliary manifest at `2e-5`. Stage 2
reloads that adapter, trains for 1.5 epochs on the 9,200 Kaggle rows at `1e-5`,
and evaluates at 0.5, 1.0, and 1.5 epochs. Only language `q/k/v/o` projections
are adapted.


In [ ]:
%%writefile train_relation_ddp.py
import argparse
import csv
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
SEED = 42
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
TARGET_NAMES = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']

SYSTEM_PROMPT = """You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.""".strip()


def shorten_caption(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + "\n[...middle truncated...]\n" + text[-half:]


def object_content(number, obj):
    if obj['kind'] == 'image':
        with Image.open(obj['value']) as source:
            image = source.convert('RGB')
        return [
            {'type': 'text', 'text': f'Object {number} is a scientific figure:'},
            {'type': 'image', 'image': image},
        ]
    return [{'type': 'text', 'text': f"Object {number} is a figure caption:\n{shorten_caption(obj['value'])}"}]


def build_messages(row, swap=False):
    obj_1, obj_2 = row['obj_1'], row['obj_2']
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({'type': 'text', 'text': 'Classify their relationship. Reply with one digit only.'})
    return [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': content},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': str(int(row['label']))}]},
    ]


class ManifestDataset(torch.utils.data.Dataset):
    def __init__(self, path, random_swap=False):
        with Path(path).open('r', encoding='utf-8') as handle:
            self.rows = [json.loads(line) for line in handle]
        self.random_swap = random_swap

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = dict(self.rows[index])
        row['_swap'] = self.random_swap and random.random() < 0.5
        return row


class LabelOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        self.label_token_ids = []
        for digit in '0123':
            ids = processor.tokenizer.encode(digit, add_special_tokens=False)
            if len(ids) != 1:
                raise ValueError(f'Label {digit} is not a single token: {ids}')
            self.label_token_ids.append(ids[0])

    def __call__(self, features):
        if len(features) != 1:
            raise ValueError(f'Expected per-device batch 1, received {len(features)}')
        row = features[0]
        batch = self.processor.apply_chat_template(
            build_messages(row, swap=row.get('_swap', False)),
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors='pt',
        )
        target_id = self.label_token_ids[int(row['label'])]
        positions = torch.where(batch['input_ids'][0] == target_id)[0]
        if not len(positions):
            raise RuntimeError('Assistant label token was not found.')
        labels = torch.full_like(batch['input_ids'], -100)
        labels[0, int(positions[-1])] = target_id
        batch['labels'] = labels
        return batch


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--stage', choices=['auxiliary', 'competition'], required=True)
    parser.add_argument('--train-manifest', required=True)
    parser.add_argument('--validation-manifest', required=True)
    parser.add_argument('--adapter-dir', required=True)
    parser.add_argument('--init-adapter')
    parser.add_argument('--work-root', required=True)
    parser.add_argument('--model-path', required=True)
    parser.add_argument('--epochs', type=float, required=True)
    parser.add_argument('--learning-rate', type=float, required=True)
    parser.add_argument('--gradient-accumulation', type=int, default=8)
    args = parser.parse_args()

    local_rank = int(os.environ.get('LOCAL_RANK', '0'))
    torch.cuda.set_device(local_rank)

    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
    from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen3VLForConditionalGeneration,
        Trainer,
        TrainerCallback,
        TrainingArguments,
    )

    random.seed(SEED + local_rank)
    np.random.seed(SEED + local_rank)
    torch.manual_seed(SEED + local_rank)

    processor = AutoProcessor.from_pretrained(args.model_path, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    processor.tokenizer.padding_side = 'right'
    collator = LabelOnlyCollator(processor)
    label_token_ids_cpu = torch.tensor(collator.label_token_ids, dtype=torch.long)
    if local_rank == 0:
        print('Label token IDs:', collator.label_token_ids, flush=True)

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        args.model_path,
        quantization_config=quantization,
        dtype=torch.float16,
        attn_implementation='sdpa',
        device_map={'': local_rank},
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    target_suffixes = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
    if args.init_adapter:
        model = PeftModel.from_pretrained(model, args.init_adapter, is_trainable=True)
        variant = f'continued from {args.init_adapter}'
    else:
        language_targets = [
            name for name, _ in model.named_modules()
            if '.visual.' not in f'.{name}.' and name.rsplit('.', 1)[-1] in target_suffixes
        ]
        if not language_targets:
            raise RuntimeError('No language LoRA targets were discovered.')
        model = get_peft_model(
            model,
            LoraConfig(
                r=16,
                lora_alpha=32,
                lora_dropout=0.05,
                bias='none',
                task_type='CAUSAL_LM',
                target_modules=language_targets,
            ),
        )
        variant = 'new language-attention adapter'

    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    if any('.visual.' in f'.{name}.' for name in trainable_names):
        raise RuntimeError('Visual parameters unexpectedly became trainable.')
    if local_rank == 0:
        print('Stage:', args.stage, '| LoRA:', variant, flush=True)
        model.print_trainable_parameters()

    class RestrictedFourClassTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            supervised = labels.ne(-100)
            answer_positions = supervised.to(torch.int64).argmax(dim=1)
            batch_indices = torch.arange(labels.shape[0], device=labels.device)
            vocabulary_logits = outputs.logits[batch_indices, answer_positions - 1]
            label_token_ids = label_token_ids_cpu.to(vocabulary_logits.device)
            class_logits = vocabulary_logits.index_select(-1, label_token_ids).float()
            target_token_ids = labels[batch_indices, answer_positions]
            matches = target_token_ids[:, None].eq(label_token_ids[None, :])
            class_targets = matches.to(torch.int64).argmax(dim=1)
            loss = F.cross_entropy(class_logits, class_targets)
            return (loss, outputs) if return_outputs else loss

    def restrict_logits_for_metrics(logits, labels):
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        supervised = labels.ne(-100)
        answer_positions = supervised.to(torch.int64).argmax(dim=1)
        batch_indices = torch.arange(labels.shape[0], device=labels.device)
        label_token_ids = label_token_ids_cpu.to(logits.device)
        return logits[batch_indices, answer_positions - 1].index_select(-1, label_token_ids)

    def targets_from_labels(labels):
        token_to_class = {token_id: index for index, token_id in enumerate(collator.label_token_ids)}
        target_token_ids = np.array([row[np.flatnonzero(row != -100)[0]] for row in labels], dtype=np.int64)
        return np.array([token_to_class[int(token_id)] for token_id in target_token_ids])

    def compute_metrics(prediction):
        logits = np.asarray(prediction.predictions)
        targets = targets_from_labels(np.asarray(prediction.label_ids))
        predictions = logits.argmax(axis=-1)
        precision, recall, f1, _ = precision_recall_fscore_support(
            targets, predictions, labels=[0, 1, 2, 3], zero_division=0
        )
        metrics = {'macro_f1': float(f1_score(targets, predictions, average='macro'))}
        for index in range(4):
            metrics[f'precision_class_{index}'] = float(precision[index])
            metrics[f'recall_class_{index}'] = float(recall[index])
            metrics[f'f1_class_{index}'] = float(f1[index])
        metrics['relation_f1'] = float((f1[1] + f1[2]) / 2)
        return metrics

    class FractionMilestoneCallback(TrainerCallback):
        def __init__(self, fractions):
            self.fractions = fractions

        def on_train_begin(self, args, state, control, **kwargs):
            self.milestones = {max(1, int(state.max_steps * fraction + 0.5)) for fraction in self.fractions}
            if state.is_world_process_zero:
                print('Evaluation/checkpoint milestones:', sorted(self.milestones), flush=True)
            return control

        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step in self.milestones:
                control.should_evaluate = True
                control.should_save = True
            return control

    train_dataset = ManifestDataset(args.train_manifest, random_swap=True)
    validation_dataset = ManifestDataset(args.validation_manifest, random_swap=False)
    competition_stage = args.stage == 'competition'
    callbacks = [FractionMilestoneCallback([1/3, 2/3, 1.0])] if competition_stage else []

    training_args = TrainingArguments(
        output_dir=str(Path(args.work_root) / 'trainer_output'),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=args.gradient_accumulation,
        num_train_epochs=args.epochs,
        learning_rate=args.learning_rate,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        logging_steps=10,
        eval_strategy='steps' if competition_stage else 'no',
        save_strategy='steps' if competition_stage else 'no',
        eval_steps=10000,
        save_steps=10000,
        save_total_limit=3,
        load_best_model_at_end=competition_stage,
        metric_for_best_model='macro_f1' if competition_stage else None,
        greater_is_better=True if competition_stage else None,
        report_to='none',
        remove_unused_columns=False,
        dataloader_num_workers=0,
        ddp_find_unused_parameters=False,
        seed=SEED,
    )
    trainer = RestrictedFourClassTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=restrict_logits_for_metrics,
        callbacks=callbacks,
    )

    torch.cuda.synchronize()
    started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    final_validation = trainer.evaluate()

    prediction_output = trainer.predict(validation_dataset) if competition_stage else None

    if trainer.is_world_process_zero():
        adapter_path = Path(args.adapter_dir)
        adapter_path.mkdir(parents=True, exist_ok=True)
        trainer.save_model(adapter_path)
        processor.save_pretrained(adapter_path)
        metrics = dict(result.metrics)
        metrics.update({f'validation_{key}': value for key, value in final_validation.items()})
        metrics.update({
            'stage': args.stage,
            'wall_minutes': elapsed / 60,
            'optimizer_steps': int(trainer.state.global_step),
            'seconds_per_optimizer_step': elapsed / max(1, trainer.state.global_step),
            'peak_gpu_gib_rank0': torch.cuda.max_memory_allocated() / 2**30,
            'best_checkpoint': trainer.state.best_model_checkpoint,
            'best_metric': trainer.state.best_metric,
            'train_rows': len(train_dataset),
            'validation_rows': len(validation_dataset),
            'learning_rate': args.learning_rate,
            'epochs': args.epochs,
            'loss_type': 'restricted_four_class_cross_entropy',
        })

        if prediction_output is not None:
            logits = np.asarray(prediction_output.predictions)
            logits = logits - logits.max(axis=1, keepdims=True)
            probabilities = np.exp(logits)
            probabilities /= probabilities.sum(axis=1, keepdims=True)
            targets = targets_from_labels(np.asarray(prediction_output.label_ids))
            predictions = probabilities.argmax(axis=1)
            matrix = confusion_matrix(targets, predictions, labels=[0, 1, 2, 3])
            metrics['confusion_matrix'] = matrix.tolist()
            modality_metrics = {}
            modalities = np.array([row['modality'] for row in validation_dataset.rows])
            for modality in sorted(set(modalities)):
                mask = modalities == modality
                modality_metrics[modality] = {
                    'rows': int(mask.sum()),
                    'macro_f1': float(f1_score(targets[mask], predictions[mask], average='macro')),
                }
            metrics['modality_metrics'] = modality_metrics

            probability_path = Path(args.work_root) / 'validation_probabilities.csv'
            with probability_path.open('w', encoding='utf-8', newline='') as handle:
                fieldnames = ['id', 'modality', 'target', 'prediction', *[f'p_{name}' for name in TARGET_NAMES]]
                writer = csv.DictWriter(handle, fieldnames=fieldnames)
                writer.writeheader()
                for index, row in enumerate(validation_dataset.rows):
                    writer.writerow({
                        'id': row['id'],
                        'modality': row['modality'],
                        'target': int(targets[index]),
                        'prediction': int(predictions[index]),
                        **{f'p_{name}': float(probabilities[index, class_index]) for class_index, name in enumerate(TARGET_NAMES)},
                    })

        with (adapter_path / 'training_metrics.json').open('w', encoding='utf-8') as handle:
            json.dump(metrics, handle, indent=2)
        print(json.dumps(metrics, indent=2), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
TRAIN_SCRIPT_PATH = Path('train_relation_ddp.py').resolve()
launch_env = dict(
    os.environ,
    PYTHONUNBUFFERED='1',
    TOKENIZERS_PARALLELISM='false',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
)


def run_stage(command, description):
    print('Launching:', ' '.join(command), flush=True)
    started = time.perf_counter()
    subprocess.run(command, check=True, env=launch_env)
    print(f'{description}: {(time.perf_counter()-started)/60:.2f} min')


if RUN_AUXILIARY_STAGE:
    auxiliary_command = [
        sys.executable, '-m', 'accelerate.commands.launch',
        '--multi_gpu', '--num_processes', '2',
        str(TRAIN_SCRIPT_PATH),
        '--stage', 'auxiliary',
        '--train-manifest', str(AUXILIARY_MANIFEST),
        '--validation-manifest', str(VALIDATION_MANIFEST),
        '--adapter-dir', str(AUXILIARY_ADAPTER),
        '--work-root', str(AUXILIARY_WORK),
        '--model-path', MODEL_PATH,
        '--epochs', '1.0',
        '--learning-rate', '2e-5',
        '--gradient-accumulation', str(GRADIENT_ACCUMULATION),
    ]
    run_stage(auxiliary_command, 'Auxiliary stage')
else:
    print('Auxiliary stage disabled; expecting adapter at', AUXILIARY_ADAPTER)

assert (AUXILIARY_ADAPTER / 'adapter_config.json').exists(), 'Auxiliary adapter is missing.'


In [ ]:
if RUN_COMPETITION_STAGE:
    competition_command = [
        sys.executable, '-m', 'accelerate.commands.launch',
        '--multi_gpu', '--num_processes', '2',
        str(TRAIN_SCRIPT_PATH),
        '--stage', 'competition',
        '--train-manifest', str(TRAIN_MANIFEST),
        '--validation-manifest', str(VALIDATION_MANIFEST),
        '--adapter-dir', str(FINAL_ADAPTER),
        '--init-adapter', str(AUXILIARY_ADAPTER),
        '--work-root', str(FINAL_WORK),
        '--model-path', MODEL_PATH,
        '--epochs', '1.5',
        '--learning-rate', '1e-5',
        '--gradient-accumulation', str(GRADIENT_ACCUMULATION),
    ]
    run_stage(competition_command, 'Competition adaptation and three validations')
else:
    print('Competition stage disabled.')

metrics_path = FINAL_ADAPTER / 'training_metrics.json'
if metrics_path.exists():
    final_metrics = json.loads(metrics_path.read_text())
    print(json.dumps(final_metrics, indent=2))
    print('Baseline macro-F1:          0.729164')
    print('Baseline same_paper F1:     0.669880')
    print('Baseline related_papers F1: 0.537897')
    print('Validation probabilities:', FINAL_WORK / 'validation_probabilities.csv')


## Promotion criteria

Scale the method only if the pilot reaches approximately:

```text
same_paper F1       >= 0.690
related_papers F1   >= 0.568
same_figure F1      >= 0.919
unrelated_papers F1 >= 0.770
macro-F1            >= 0.739
```

The required artifacts are `best_relation_adapter/training_metrics.json` and
`competition_stage/validation_probabilities.csv`. If the pilot succeeds, the
next experiment should scale the same construction to 50K–100K pairs and add
hard-example mining as a separate ablation.
